# Lektion 13 - Agentminne med Cognee Knowledge Graphs


## Installera

Detta anteckningsbok demonstrerar hur man bygger en intelligent **kodningsassistent** med bestående minne med hjälp av [**Cognee**](https://www.cognee.ai/) kunskapsgrafer och **Microsoft Agent Framework** (MAF).

Cognee förvandlar ostrukturerad text till en strukturerad, frågbar kunskapsgraf som stöds av vektorinkapslingar — vilket ger din agent ett rikt, relationsmedvetet långtidsminne.

### Vad du kommer att lära dig
1. **Bygga kunskapsgrafer**: Omvandla utvecklarprofiler och bästa praxis till strukturerad, frågbar kunskap.
2. **Integrera Cognee med MAF**: Använd `@tool`-funktioner för att låta en MAF-agent fråga Cognees kunskapsgraf.
3. **Sessionsmedvetna samtal**: Behåll kontext över flera frågor i samma session.
4. **Långtidsminne**: Behåll viktig kunskap över sessioner och hämta den i nya samtal.

### Förutsättningar
- Python 3.9+
- Redis som körs lokalt (`docker run -d -p 6379:6379 redis`) för sessionshantering
- En LLM API-nyckel (t.ex. OpenAI) — sätt `LLM_API_KEY` i `.env`
- `CACHING=true` i `.env` (krävs för Cognee-sessioner)
- Ett Azure AI Foundry-projekt med en distribuerad chattmodell
- `AZURE_AI_PROJECT_ENDPOINT` och `AZURE_AI_MODEL_DEPLOYMENT_NAME` i `.env`
- Azure CLI autentiserad (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typer av agentminnen

Denna anteckningsbok utforskar samma tre minnestyper från huvudboken Lektion 13, men använder Cognee som backend för långtidsminnet:

| Memory Type | Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` (MAF) | Enskild konversation |
| **Short-term** | Cognee session cache (Redis) | Enskild session |
| **Long-term** | Cognee knowledge graph + vectors | Permanent |

### Cognees minnesarkitektur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Förbered Cognee-lagring


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — Bygga kunskapsbasen

Vi tar in tre typer av data för att skapa en omfattande kunskapsbas för vår kodningsassistent:

1. **Utvecklarprofil** — personlig expertis och teknisk bakgrund
2. **Python bästa praxis** — Python Zen med praktiska riktlinjer
3. **Historiska konversationer** — tidigare frågor och svar mellan utvecklare och AI-assistenter


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualisera kunskapsgrafen

Cognee kan visa en interaktiv HTML-visualisering av de entiteter och relationer som den har extraherat.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Berika minnet med Memify

`memify()` analyserar kunskapsgrafen och genererar intelligenta regler — identifierar mönster, bästa praxis och relationer mellan begrepp.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Del 2 — MAF-agent med Cognee-verktyg

Nu skapar vi en MAF-agent som kan fråga Cognees kunskapsgraf via `@tool`-funktioner. Detta låter agenten utnyttja den fulla kraften av grafmedveten semantisk sökning samtidigt som den behåller kontexten i samtalet genom sessioner.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Arbetsminne med sessioner

`AgentSession` (skapad via `agent.create_session()`) tillhandahåller arbetsminne inom en session. agenten kan referera tillbaka till tidigare meddelanden samtidigt som den även kan fråga Cognees långsiktiga kunskapsgraf.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Ny session — Långtidsminnet består

Att starta en ny session rensar arbetsminnet, men Cognee kunskapsgraf finns fortfarande tillgänglig. Agenten kan hämta samma långtidskunskap i en helt ny konversation.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sammanfattning

I denna anteckningsbok byggde du en kodningsassistent som kombinerar **MAF:s arbetsminne** (`agent.create_session()`) med **Cognees långsiktiga kunskapsgraf**.

### Vad du har lärt dig
1. **Kunskapsgrafkonstruktion**: Cognee bearbetar ostrukturerad text och bygger en graf + vektorminne.
2. **Grafberikning med memify**: Härledda fakta och rikare relationer ovanpå din befintliga graf.
3. **MAF + Cognee integration**: `@tool`-funktioner låter en MAF-agent fråga Cognees graf på ett naturligt sätt.
4. **Arbetsminne + långsiktigt minne**: `AgentSession` (via `agent.create_session()`) ger sessionskontext medan Cognee tillhandahåller bestående kunskap.
5. **Filtrerad sökning med NodeSets**: Rikta in dig på specifika delmängder av kunskapsgrafen (t.ex. endast principer).

### Viktiga insikter
- **Cognee** omvandlar rå text till strukturerat, relationsmedvetet minne — kraftfullare än ett platt vektorarkiv.
- **`@tool`-funktioner** kopplar samman MAF-agenter och externa kunskapssystem på ett rent sätt.
- **`AgentSession`** (via `agent.create_session()`) håller per-konversationskontext åtskild från långlivad kunskap.
- Samma kunskapsgraf tjänar flera sessioner och agenter.

### Verkliga användningsområden
- **Utvecklar-assistenter**: Kodgranskning, incidentanalys, arkitektassistenter
- **Kundnära assistenter**: Supportagenter över produktdokumentation, vanliga frågor och CRM-anteckningar
- **Interna expertassistenter**: Policy-, juridik- eller säkerhetsassistenter som resonerar kring riktlinjer
- **Enade datalager**: Kombinera strukturerade och ostrukturerade data i en och samma frågebar graf

### Nästa steg
- Experimentera med temporal medvetenhet i Cognee
- Definiera en OWL-ontologi för domänspecifik grafkvalitet
- Lägg till användarfeedback-loopar för att förbättra hämtning över tid
- Skala upp till multi-agent-system som delar samma Cognee-minneslager


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, bör du vara medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår vid användning av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
